In [ ]:
import pandas as pd

In [ ]:
IN_CSV = R'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\Elicit\papers_tables\Elicit retrieved.csv'
IN_RIS = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\Paul\articles.ris'
OUT_FILE = R'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\Elicit\papers_tables\Elicit retrieved 2.csv'

In [ ]:
df = pd.read_csv(IN_CSV)
df.head()

In [ ]:
entry = {'Year': None, 'Authors': [], 'Title': None}
ris_entries = []
with open(IN_RIS, 'rt', encoding="utf-8") as f:
    for line in f.readlines():
        if not line.strip():
            if entry['Year'] or entry['Authors'] or entry['Title']:
                entry['Authors'] = '\n'.join(entry['Authors'])
                ris_entries.append(entry)
            entry = {'Year': None, 'Authors': [], 'Title': None}
            continue

        typ, text = line.split('-', 1)
        typ = typ.strip()
        text = text.strip()
        if typ == 'TI':
            entry['Title'] = text
        elif typ == 'Y1':
            entry['Year'] = int(text)
        elif typ == 'AU':
            entry['Authors'].append(text)

    if entry['Year'] or entry['Authors'] or entry['Title']:
        entry['Authors'] = '\n'.join(sorted(entry['Authors']))
        ris_entries.append(entry)

ris_df = pd.DataFrame(ris_entries)
ris_df.head()

In [ ]:
df['in_csv'] = True

In [ ]:
len(ris_df), len(ris_df.drop_duplicates())

In [ ]:
ris_df.drop_duplicates(inplace=True)

In [ ]:
ris_titles = set(ris_df['Title'])
csv_titles = set(df['Title'])
len(ris_titles), len(csv_titles), len(ris_titles & csv_titles)

In [ ]:
len(df)

In [ ]:
lower_ris_titles = [t.lower() for t in ris_titles]
lower_csv_titles = [t.lower() for t in csv_titles]

In [ ]:
df['in_ris'] = df['Title'].str.lower().isin(lower_ris_titles)
df.in_ris.sum()

In [ ]:
ris_df['in_csv'] = ris_df['Title'].str.lower().isin(lower_csv_titles)
ris_df['in_ris'] = True
ris_df = ris_df[~ris_df.in_csv]

In [ ]:
combined = pd.concat([df, ris_df], axis=0)

In [ ]:
combined.to_csv(OUT_FILE, index=False)